In [1]:
%load_ext watermark
%watermark -a "Manuel Elias Orellana Lavayen" -d -v -iv

Author: Manuel Elias Orellana Lavayen

Date: 2026-08-29

Python implementation: CPython
Python version       : 3.12.10
IPython version      : 9.17.0



# Librerias

In [2]:
%load_ext cuml.accel
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB,ComplementNB, GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np
from scipy.sparse import load_npz
from pathlib import Path
ruta_base = Path.cwd() #Ruta base

In [3]:
import cuml
import rmm
cuml.accel.install(log_level="info") # Mostrara informacion sobre el acelerador
rmm.reinitialize(managed_memory=False)# Configurar uso exclusivo de la memoria Vram

# Cargar Matrices y Embeddings

### Función para cargar Matrices TF-IDF de manera de diccionario

In [4]:
def cargar_vectorizaciones_tfidf(ruta_base):
    """
    Cargar de manera de diccionario las vectorizaciones de la ruta

    Args:
        ruta_base: Hace referencia a carpeta donde se encuentra el notebook

    """
    ruta_base = Path(ruta_base)
    resultados = {}

    # Sublinear
    for carpeta_sublinear in ruta_base.iterdir():
        if not carpeta_sublinear.is_dir():
            continue
        nombre_sublinear = carpeta_sublinear.name
        resultados[nombre_sublinear] = {}

        # N-gramas
        for carpeta_ngram in carpeta_sublinear.iterdir():
            if not carpeta_ngram.is_dir():
                continue
            nombre_ngram = carpeta_ngram.name
            resultados[nombre_sublinear][nombre_ngram] = {}

            # Max features
            for carpeta_features in carpeta_ngram.iterdir():
                if not carpeta_features.is_dir():
                    continue
                nombre_features = carpeta_features.name

                # Rutas
                ruta_train = (carpeta_features /"X_train_tfidf.npz")
                ruta_validation = (carpeta_features /"X_validation_tfidf.npz")
                ruta_test = (carpeta_features /"X_test_tfidf.npz")

                # Comprobar que existan
                if not (ruta_train.exists() and ruta_validation.exists()and ruta_test.exists()):
                    print(f"Faltan archivos en: "f"{carpeta_features}")
                    continue

                # Cargar matrices
                X_train = load_npz(ruta_train)
                X_validation = load_npz(ruta_validation)
                X_test = load_npz(ruta_test)

                # Convertir a float32
                X_train = X_train.astype(np.float32)
                X_validation = X_validation.astype(np.float32)
                X_test = X_test.astype(np.float32)

                # Guardar
                resultados[nombre_sublinear][nombre_ngram][nombre_features] = {"X_train": X_train,"X_validation": X_validation,"X_test": X_test}
                print(
                    f"Cargado: "
                    f"{nombre_sublinear} | "
                    f"{nombre_ngram} | "
                    f"{nombre_features}"
                )

    return resultados

### Cargando Matrices TF-IDF

In [5]:
dict_vectorizacion_tfidf_normal = cargar_vectorizaciones_tfidf("./matrices y embeddings/No Ortografia/Matrices TF-IDF")
dict_vectorizacion_tfidf_ortografia = cargar_vectorizaciones_tfidf("./matrices y embeddings/Ortografia/Matrices TF-IDF")

Cargado: sublinear_false | ngram_2 | 10000
Cargado: sublinear_false | ngram_2 | 20000
Cargado: sublinear_false | ngram_2 | 30000
Cargado: sublinear_false | ngram_2 | 40000
Cargado: sublinear_false | ngram_2 | 50000
Cargado: sublinear_false | ngram_3 | 10000
Cargado: sublinear_false | ngram_3 | 20000
Cargado: sublinear_false | ngram_3 | 30000
Cargado: sublinear_false | ngram_3 | 40000
Cargado: sublinear_false | ngram_3 | 50000
Cargado: sublinear_false | ngram_4 | 10000
Cargado: sublinear_false | ngram_4 | 20000
Cargado: sublinear_false | ngram_4 | 30000
Cargado: sublinear_false | ngram_4 | 40000
Cargado: sublinear_false | ngram_4 | 50000
Cargado: sublinear_true | ngram_2 | 10000
Cargado: sublinear_true | ngram_2 | 20000
Cargado: sublinear_true | ngram_2 | 30000
Cargado: sublinear_true | ngram_2 | 40000
Cargado: sublinear_true | ngram_2 | 50000
Cargado: sublinear_true | ngram_3 | 10000
Cargado: sublinear_true | ngram_3 | 20000
Cargado: sublinear_true | ngram_3 | 30000
Cargado: sublinear_

### Cargando Embeddings

In [6]:
X_train_embeddings_vectorizados_normal = (np.load("./matrices y embeddings/No Ortografia/Embeddings/X_train_embeddings.npy")).astype(np.float32)
X_validation_embeddings_vectorizados_normal = (np.load("./matrices y embeddings/No Ortografia/Embeddings/X_validation_embeddings.npy")).astype(np.float32)
X_test_embeddings_vectorizados_normal = (np.load("./matrices y embeddings/No Ortografia/Embeddings/X_test_embeddings.npy")).astype(np.float32)

X_train_embeddings_vectorizados_ortografia = (np.load("./matrices y embeddings/Ortografia/Embeddings/X_train_embeddings.npy")).astype(np.float32)
X_validation_embeddings_vectorizados_ortografia = (np.load("./matrices y embeddings/Ortografia/Embeddings/X_validation_embeddings.npy")).astype(np.float32)
X_test_embeddings_vectorizados_ortografia = (np.load("./matrices y embeddings/Ortografia/Embeddings/X_test_embeddings.npy")).astype(np.float32)

dict_embeddings = {
    "Ortografia":{
        "X_train": X_train_embeddings_vectorizados_normal,
        "X_validation": X_validation_embeddings_vectorizados_normal,
        "X_test": X_test_embeddings_vectorizados_normal,
    },
    "No Ortografia":{
        "X_train": X_train_embeddings_vectorizados_ortografia,
        "X_validation": X_validation_embeddings_vectorizados_ortografia,
        "X_test": X_test_embeddings_vectorizados_ortografia
    }
}

### Cargando Y

In [7]:
y_train = (np.load("./Y/y_train.npy")).astype(np.int32)
y_validation = (np.load("./Y/y_validation.npy")).astype(np.int32)
y_test = (np.load("./Y/y_test.npy")).astype(np.int32)

## Entrenamiento Automatizado

### Funciones para entrenamiento Automatizado

In [8]:
#Función para calcular metricas de varios modelos
def metricas_clasificacion(modelos: dict):
    """
    Calcula las metricas de clasificacion de los modelos que se encuentran en el diccionario

    Args:
        modelos: diccionario de modelos con su respectivo y_pred y y_true

    Returns:
        Dataframe comparativo con las metricas de los modelos
    """
    lista_metricas = []
    for nombre, datos in modelos.items():
        y_true = datos["y_true"]
        y_pred = datos["y_pred"]

        exactitud = accuracy_score(y_true, y_pred)
        precision_macro = precision_score(y_true,y_pred,average="macro",zero_division=0)
        recall_macro = recall_score(y_true,y_pred,average="macro",zero_division=0)
        f1_macro = f1_score(y_true,y_pred,average="macro",zero_division=0)
        f1_weighted = f1_score(y_true,y_pred,average="weighted",zero_division=0)

        lista_metricas.append({
            "Modelo": nombre,
            "Exactitud": exactitud,
            "Precision Macro": precision_macro,
            "Recall Macro": recall_macro,
            "F1 Macro": f1_macro,
            "F1 Weighted": f1_weighted
        })
    tabla_metricas_modelos = pd.DataFrame(lista_metricas)
    return tabla_metricas_modelos

#Función para entrenar valios modelos y calcular sus metricas
def entrenamiento_modelos(
    X_train,
    X_validation,
    y_train,
    y_validation,
    modelos,
    representacion,
    ortografia,
    vectores_tfidf = True,
    sublinear=None,
    ngram=None,
    max_features=None
):
    """
    Entrena los modelos que recibe del diccionario con los valores X_train y y_train y luego con ayuda de la funcion metricas_clasificacion obtiene una tabla comparativa con mas métricas de clasificación de los modelos, ademas esta tabla comparativa posee informacion del vectorizador, lo que resulta util en la seleccion de modelos

    Args:
        X_train: Vectores TF-IDF o Embeddings de entrenamiento
        X_validation: Vectores TF-IDF o Embeddings de validación
        y_train: Array de Numpy con valores predictores para train
        y_validation: Array de Numpy con valores predictores para validation
        modelos: Diccionario de Modelos sin entrenar
        representacion: Nombre de la representacion
        ortografia: Booleano de ortografia
        vectores_tfidf = True : Booleano de entrenamiento con vectores TF-IDF
        sublinear=None : Especifica el sublinear de la vectorizacion
        ngram=None : Especifica el ngram de la vectorizacion
        max_features=None : Especifica el max_feature de la vectorizacion

    Returns:
        Tabla de metricas de los modelos entrenados
    """

    resultados_modelos = {}
    for nombre_modelo, modelo in modelos.items():
        print(f"Entrenando: {nombre_modelo}")

        # Entrenamiento
        modelo.fit(X_train,y_train)

        # Predicción
        y_pred = modelo.predict(X_validation)
        # Probabilidades
        if hasattr(modelo, "predict_proba"): #Verificar que el modelo tenga la función de pred_proba
            y_proba = modelo.predict_proba(X_validation)
        else:
            y_proba = None

        # Guardar resultados para metricas_clasificacion()
        resultados_modelos[nombre_modelo] = {"y_true": y_validation,"y_pred": y_pred,"y_proba": y_proba}

    # Calcular métricas
    tabla_metricas = metricas_clasificacion(resultados_modelos)

    # Agregar información de la representación
    tabla_metricas["Representacion"] = representacion
    tabla_metricas["Ortografia"] = ortografia

    if vectores_tfidf:
        tabla_metricas["Sublinear"] = sublinear
        tabla_metricas["Ngram"] = ngram
        tabla_metricas["Max Features"] = max_features

        # Reordenar columnas
        tabla_metricas = tabla_metricas[
            [
                "Representacion",
                "Ortografia",
                "Sublinear",
                "Ngram",
                "Max Features",
                "Modelo",
                "Exactitud",
                "Precision Macro",
                "Recall Macro",
                "F1 Macro",
                "F1 Weighted"
            ]
        ]
    else:
        # Reordenar columnas
        tabla_metricas = tabla_metricas[
            [
                "Representacion",
                "Ortografia",
                "Modelo",
                "Exactitud",
                "Precision Macro",
                "Recall Macro",
                "F1 Macro",
                "F1 Weighted"
            ]
        ]

    return tabla_metricas

# Modelos TF-IDF

## Diccionario de Modelos a usarse para TF-IDF

In [9]:
modelos = {
    "LogisticRegression": LogisticRegression(
        max_iter=3000,
        random_state=30,
        class_weight='balanced'
    ),

    "LinearSVC": LinearSVC(
        random_state=30,
        class_weight='balanced'
    ),

    "Multinomial NaiveBayes": MultinomialNB(),

    "Compelment NaiveBayes": ComplementNB()
}

### Entrenamiento de modelos con vectores TF-IDF sin correción ortografica

In [10]:
lista_resultados_tfidf_normal = []

for sublinear, datos_sublinear in dict_vectorizacion_tfidf_normal.items():
    for ngram, datos_ngram in datos_sublinear.items():
        for max_features, datos in datos_ngram.items():
            print(
                f"\n{'=' * 60}"
                f"\nConfiguración:"
                f"\nSublinear: {sublinear}"
                f"\nN-grama: {ngram}"
                f"\nMax features: {max_features}"
                f"\n{'=' * 60}"
            )

            tabla = entrenamiento_modelos(
                X_train=datos["X_train"],
                X_validation=datos["X_validation"],
                y_train=y_train,
                y_validation=y_validation,
                modelos=modelos,

                representacion="TF-IDF",
                ortografia=False,

                sublinear=sublinear,
                ngram=ngram,
                max_features=max_features
            )

            lista_resultados_tfidf_normal.append(tabla)

resultados_tfidf_normal = pd.concat(lista_resultados_tfidf_normal,ignore_index=True)


Configuración:
Sublinear: sublinear_false
N-grama: ngram_2
Max features: 10000
Entrenando: LogisticRegression
Entrenando: LinearSVC
Entrenando: Multinomial NaiveBayes
Entrenando: Compelment NaiveBayes

Configuración:
Sublinear: sublinear_false
N-grama: ngram_2
Max features: 20000
Entrenando: LogisticRegression
Entrenando: LinearSVC
Entrenando: Multinomial NaiveBayes
Entrenando: Compelment NaiveBayes

Configuración:
Sublinear: sublinear_false
N-grama: ngram_2
Max features: 30000
Entrenando: LogisticRegression
Entrenando: LinearSVC
Entrenando: Multinomial NaiveBayes
Entrenando: Compelment NaiveBayes

Configuración:
Sublinear: sublinear_false
N-grama: ngram_2
Max features: 40000
Entrenando: LogisticRegression
Entrenando: LinearSVC
Entrenando: Multinomial NaiveBayes
Entrenando: Compelment NaiveBayes

Configuración:
Sublinear: sublinear_false
N-grama: ngram_2
Max features: 50000
Entrenando: LogisticRegression
Entrenando: LinearSVC
Entrenando: Multinomial NaiveBayes
Entrenando: Compelment N

### Entrenamiento de modelos con vectores TF-IDF con  correción ortografica

In [11]:
lista_resultados_tfidf_ortografia = []

for sublinear, datos_sublinear in dict_vectorizacion_tfidf_ortografia.items():
    for ngram, datos_ngram in datos_sublinear.items():
        for max_features, datos in datos_ngram.items():
            print(
                f"\n{'=' * 60}"
                f"\nConfiguración:"
                f"\nSublinear: {sublinear}"
                f"\nN-grama: {ngram}"
                f"\nMax features: {max_features}"
                f"\n{'=' * 60}"
            )

            tabla = entrenamiento_modelos(
                X_train=datos["X_train"],
                X_validation=datos["X_validation"],
                y_train=y_train,
                y_validation=y_validation,
                modelos=modelos,

                representacion="TF-IDF",
                ortografia=True,

                sublinear=sublinear,
                ngram=ngram,
                max_features=max_features
            )

            lista_resultados_tfidf_ortografia.append(tabla)

resultados_tfidf_ortografia = pd.concat(lista_resultados_tfidf_ortografia,ignore_index=True)


Configuración:
Sublinear: sublinear_false
N-grama: ngram_2
Max features: 10000
Entrenando: LogisticRegression
Entrenando: LinearSVC
Entrenando: Multinomial NaiveBayes
Entrenando: Compelment NaiveBayes

Configuración:
Sublinear: sublinear_false
N-grama: ngram_2
Max features: 20000
Entrenando: LogisticRegression
Entrenando: LinearSVC
Entrenando: Multinomial NaiveBayes
Entrenando: Compelment NaiveBayes

Configuración:
Sublinear: sublinear_false
N-grama: ngram_2
Max features: 30000
Entrenando: LogisticRegression
Entrenando: LinearSVC
Entrenando: Multinomial NaiveBayes
Entrenando: Compelment NaiveBayes

Configuración:
Sublinear: sublinear_false
N-grama: ngram_2
Max features: 40000
Entrenando: LogisticRegression
Entrenando: LinearSVC
Entrenando: Multinomial NaiveBayes
Entrenando: Compelment NaiveBayes

Configuración:
Sublinear: sublinear_false
N-grama: ngram_2
Max features: 50000
Entrenando: LogisticRegression
Entrenando: LinearSVC
Entrenando: Multinomial NaiveBayes
Entrenando: Compelment N

# Modelos Embeddigs

## Diccionario de Modelos a usarse para Embeddings

In [12]:
modelos_embeddings = {

    "LogisticRegression": LogisticRegression(
        max_iter=3000,
        random_state=30,
        class_weight='balanced'
    ),

    "LinearSVC": LinearSVC(
        random_state=30,
        class_weight='balanced'
    ),

    "Gausian NaiveBayes": GaussianNB(),
}

### Entrenamiento de modelos con vectores Embeddings con correción ortografica

In [13]:
resultados_embeddings_ortografia = entrenamiento_modelos(
    X_train=dict_embeddings["Ortografia"]["X_train"],
    X_validation=dict_embeddings["Ortografia"]["X_validation"],
    y_train=y_train,
    y_validation=y_validation,
    modelos=modelos_embeddings,
    ortografia="Si",
    vectores_tfidf= False,
    representacion= "Embeddings"
)

Entrenando: LogisticRegression
Entrenando: LinearSVC
Entrenando: Gausian NaiveBayes


### Entrenamiento de modelos con vectores TF-IDF sin correción ortografica

In [14]:
resultados_embeddings_normal = entrenamiento_modelos(
    X_train=dict_embeddings["No Ortografia"]["X_train"],
    X_validation=dict_embeddings["No Ortografia"]["X_validation"],
    y_train=y_train,
    y_validation=y_validation,
    modelos=modelos_embeddings,
    ortografia="No",
    vectores_tfidf= False,
    representacion= "Embeddings"
)

Entrenando: LogisticRegression
Entrenando: LinearSVC
Entrenando: Gausian NaiveBayes


# Resultado de Modelos

In [15]:
resultados_tfidf_normal.sort_values(by="Exactitud",ascending=False)

,Representacion,Ortografia,Sublinear,Ngram,Max Features,Modelo,Exactitud,Precision Macro,Recall Macro,F1 Macro,F1 Weighted
44,TF-IDF,False,sublinear_false,ngram_4,20000,LogisticRegression,0.725871,0.659552,0.645519,0.639099,0.702891
24,TF-IDF,False,sublinear_false,ngram_3,20000,LogisticRegression,0.725871,0.659790,0.645852,0.639631,0.703172
4,TF-IDF,False,sublinear_false,ngram_2,20000,LogisticRegression,0.724469,0.659709,0.644188,0.637774,0.701183
12,TF-IDF,False,sublinear_false,ngram_2,40000,LogisticRegression,0.724469,0.659831,0.644353,0.638047,0.701332
36,TF-IDF,False,sublinear_false,ngram_3,50000,LogisticRegression,0.723668,0.655495,0.643351,0.636753,0.700932
...,...,...,...,...,...,...,...,...,...,...,...
93,TF-IDF,False,sublinear_true,ngram_3,40000,LinearSVC,0.695435,0.633841,0.634300,0.633480,0.690313
97,TF-IDF,False,sublinear_true,ngram_3,50000,LinearSVC,0.695435,0.633841,0.634300,0.633480,0.690313
109,TF-IDF,False,sublinear_true,ngram_4,30000,LinearSVC,0.694634,0.633095,0.633633,0.632829,0.689753
113,TF-IDF,False,sublinear_true,ngram_4,40000,LinearSVC,0.694634,0.633095,0.633633,0.632829,0.689753


In [16]:
resultados_tfidf_ortografia.sort_values(by="Exactitud",ascending=False)

,Representacion,Ortografia,Sublinear,Ngram,Max Features,Modelo,Exactitud,Precision Macro,Recall Macro,F1 Macro,F1 Weighted
72,TF-IDF,True,sublinear_true,ngram_2,40000,LogisticRegression,0.722267,0.654257,0.641681,0.634862,0.699161
68,TF-IDF,True,sublinear_true,ngram_2,30000,LogisticRegression,0.720865,0.652356,0.640347,0.633463,0.697827
4,TF-IDF,True,sublinear_false,ngram_2,20000,LogisticRegression,0.720665,0.651595,0.639682,0.632479,0.697231
8,TF-IDF,True,sublinear_false,ngram_2,30000,LogisticRegression,0.720465,0.652267,0.639847,0.632885,0.697234
76,TF-IDF,True,sublinear_true,ngram_2,50000,LogisticRegression,0.720465,0.651974,0.640512,0.633942,0.697985
...,...,...,...,...,...,...,...,...,...,...,...
109,TF-IDF,True,sublinear_true,ngram_4,30000,LinearSVC,0.693232,0.632821,0.633304,0.632593,0.688891
117,TF-IDF,True,sublinear_true,ngram_4,50000,LinearSVC,0.693232,0.632821,0.633304,0.632593,0.688891
49,TF-IDF,True,sublinear_false,ngram_4,30000,LinearSVC,0.692831,0.632733,0.633134,0.632467,0.688457
53,TF-IDF,True,sublinear_false,ngram_4,40000,LinearSVC,0.692831,0.632733,0.633134,0.632467,0.688457


In [17]:
resultados_embeddings_normal.sort_values(by="Exactitud",ascending=False)

,Representacion,Ortografia,Modelo,Exactitud,Precision Macro,Recall Macro,F1 Macro,F1 Weighted
0,Embeddings,No,LogisticRegression,0.750701,0.685781,0.672036,0.667911,0.730584
1,Embeddings,No,LinearSVC,0.720865,0.677299,0.677469,0.676868,0.724926
2,Embeddings,No,Gausian NaiveBayes,0.698038,0.674929,0.669074,0.666185,0.711954


In [18]:
resultados_embeddings_ortografia.sort_values(by="Exactitud",ascending=False)

,Representacion,Ortografia,Modelo,Exactitud,Precision Macro,Recall Macro,F1 Macro,F1 Weighted
0,Embeddings,Si,LogisticRegression,0.750300,0.684573,0.671541,0.667324,0.730263
1,Embeddings,Si,LinearSVC,0.725671,0.678083,0.677976,0.677905,0.727379
2,Embeddings,Si,Gausian NaiveBayes,0.700040,0.677140,0.671740,0.668540,0.713857


# Guardar Resultados

In [19]:
resultados_tfidf_normal.to_csv(path_or_buf= ruta_base / "Resultados Modelos Experimentos/resultados_tfidf_normal.csv")
resultados_tfidf_ortografia.to_csv(path_or_buf= ruta_base / "Resultados Modelos Experimentos/resultados_tfidf_ortografia.csv")
resultados_embeddings_normal.to_csv(path_or_buf= ruta_base / "Resultados Modelos Experimentos/resultados_embeddings_normal.csv")
resultados_embeddings_ortografia.to_csv(path_or_buf= ruta_base / "Resultados Modelos Experimentos/resultados_embeddings_ortografia.csv")